# Event segmentation and onset detection

This notebook implements the event segmentation pipeline for identifying temporal windows in seismic signals. The pipeline consists of:

1. **Crustal velocity estimation** — Average P-wave and S-wave velocities are computed for each station using the CRUST1.0 global model (Laske et al., 2013)
2. **Theoretical arrival times** — P and S wave theoretical arrivals are calculated using epicentral distances and crustal velocities
3. **Signal conversion** — Long-format acceleration DataFrame is converted to nested dictionary structure for efficient processing
4. **Onset detection** — AR-AIC method (ObsPy) detects P and S onsets using all 3 components simultaneously
5. **Temporal windowing** — Four dynamical regimes are defined: pre-arrival, P-wave, S-wave, and coda

The output is a DataFrame with one row per file (66 rows = 22 stations × 3 components) containing detected onset times and window boundaries for moment scaling analysis.

## 1. Imports and visualization settings

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent / 'src'))
import pandas as pd
import numpy as np
import logging
from IPython.display import display
from src import (
    convert_signals_to_dict,
    validate_signals_dict,
    add_crustal_velocities,
    add_theoretical_arrivals,
    set_plot_style
)
colors = set_plot_style()
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger()
def check(condition, message):
    if condition:
        logger.info(message)
    else:
        raise ValueError(message)
logger.info("Environment ready")

INFO | Environment ready


## 2. Data loading

Preprocessed metadata and acceleration signals are loaded from parquet files. The metadata contains 66 rows (22 stations × 3 components), while the signals DataFrame is in long format (one row per sample).

In [2]:
# Paths
METADATA_PATH = Path('../data/processed/01_metadata/metadata_clean.parquet')
SIGNALS_PATH = Path('../data/processed/02_signals/acc_preprocessed_scaling.parquet')
OUTPUT_DIR = Path('../data/processed/04_segmentation')

# Check paths exist
check(METADATA_PATH.exists(), f"Metadata file found: {METADATA_PATH}")
check(SIGNALS_PATH.exists(), f"Signals file found: {SIGNALS_PATH}")

# Load metadata
logger.info("Loading metadata...")
df_meta = pd.read_parquet(METADATA_PATH)
check(df_meta is not None, "Metadata loaded successfully")
check(len(df_meta) > 0, "Metadata dataframe is not empty")
logger.info(f"Metadata loaded, shape: {df_meta.shape}")

# Load signals
logger.info("Loading acceleration signals...")
df_signals = pd.read_parquet(SIGNALS_PATH)
check(df_signals is not None, "Signals loaded successfully")
check(len(df_signals) > 0, "Signals dataframe is not empty")
logger.info(f"Signals loaded, shape: {df_signals.shape}")
logger.info(f"Unique files: {df_signals['file'].nunique()}")

INFO | Metadata file found: ../data/processed/01_metadata/metadata_clean.parquet
INFO | Signals file found: ../data/processed/02_signals/acc_preprocessed_scaling.parquet
INFO | Loading metadata...
INFO | Metadata loaded successfully
INFO | Metadata dataframe is not empty
INFO | Metadata loaded, shape: (66, 38)
INFO | Loading acceleration signals...
INFO | Signals loaded successfully
INFO | Signals dataframe is not empty
INFO | Signals loaded, shape: (2614815, 3)
INFO | Unique files: 66


## 3. Metadata preparation

The metadata DataFrame contains 3 rows per station (one for each component: HNE, HNN, HNZ). For crustal velocity estimation and theoretical arrival calculations, we reduce this to 1 row per station, keeping only the essential columns.

In [3]:
logger.info("Preparing station metadata (1 row per station)...")

# Select essential columns and reduce to 1 row per station
df_meta_stations = df_meta.drop_duplicates('STATION_CODE')[[
    'STATION_CODE',
    'STATION_LATITUDE_DEGREE',
    'STATION_LONGITUDE_DEGREE',
    'EPICENTRAL_DISTANCE_KM',
    'INSTRUMENTAL_FREQUENCY_HZ',
    'LOW_CUT_FREQUENCY_HZ',
    'HIGH_CUT_FREQUENCY_HZ',
    'PGA_CM/S^2',
    'TIME_PGA_S'
]].copy()

n_stations = len(df_meta_stations)
logger.info(f"Station metadata ready: {n_stations} unique stations")

# Display first rows
print("\nFirst 5 stations:")
display(df_meta_stations.head())

# Summary statistics
print("\nEpicentral distance range:")
print(f"  Min: {df_meta_stations['EPICENTRAL_DISTANCE_KM'].min():.2f} km")
print(f"  Max: {df_meta_stations['EPICENTRAL_DISTANCE_KM'].max():.2f} km")
print(f"  Median: {df_meta_stations['EPICENTRAL_DISTANCE_KM'].median():.2f} km")

INFO | Preparing station metadata (1 row per station)...
INFO | Station metadata ready: 22 unique stations



First 5 stations:


,STATION_CODE,STATION_LATITUDE_DEGREE,STATION_LONGITUDE_DEGREE,EPICENTRAL_DISTANCE_KM,INSTRUMENTAL_FREQUENCY_HZ,LOW_CUT_FREQUENCY_HZ,HIGH_CUT_FREQUENCY_HZ,PGA_CM/S^2,TIME_PGA_S
0,EILF,43.547900,7.131200,109.5,200.0,0.5,40.0,0.376358,78.870
3,ESCA,43.831000,7.374400,86.5,200.0,0.5,40.0,0.483198,73.185
6,ISO,44.184000,7.050000,39.8,200.0,0.4,40.0,0.547306,67.950
9,MFC,43.967022,6.919558,60.9,200.0,0.4,40.0,0.339241,71.770
12,MON,43.730343,7.424688,98.2,200.0,0.5,20.0,0.114282,75.935



Epicentral distance range:
  Min: 4.80 km
  Max: 109.50 km
  Median: 67.80 km


## 4. Crustal velocity estimation

Average P-wave and S-wave velocities are computed for each station location using the CRUST1.0 global crustal model. The model provides velocities for 9 layers (water, ice, 3 sediment layers, 3 crystalline crust layers, mantle) on a 1°×1° grid. We average the velocities across the three crystalline crust layers (upper, middle, lower) to obtain representative crustal velocities.

### References

Laske, G., Masters, G., Ma, Z., & Pasyanos, M. (2013). Update on CRUST1.0 - A 1-degree global model of Earth's crust. *Geophysical Research Abstracts*, 15, EGU2013-2658.

In [4]:
# Add crustal velocities (vp_crust, vs_crust)
df_meta_stations = add_crustal_velocities(
    df_meta_stations,
    lat_col='STATION_LATITUDE_DEGREE',
    lon_col='STATION_LONGITUDE_DEGREE'
)

check('vp_crust' in df_meta_stations.columns, "vp_crust column added")
check('vs_crust' in df_meta_stations.columns, "vs_crust column added")

# Display results
print("\nCrustal velocities (first 5 stations):")
display(df_meta_stations[[
    'STATION_CODE',
    'EPICENTRAL_DISTANCE_KM',
    'vp_crust',
    'vs_crust'
]].head())

INFO | vp_crust column added
INFO | vs_crust column added


Loading CRUST1.0 model...
Querying 22 stations...
Added vp_crust and vs_crust columns
v_P: min=6.33, max=6.73, median=6.33 km/s
v_S: min=3.60, max=3.79, median=3.66 km/s

Crustal velocities (first 5 stations):


,STATION_CODE,EPICENTRAL_DISTANCE_KM,vp_crust,vs_crust
0,EILF,109.5,6.729618,3.789856
3,ESCA,86.5,6.729618,3.789856
6,ISO,39.8,6.331176,3.663886
9,MFC,60.9,6.330868,3.663735
12,MON,98.2,6.729618,3.789856


## 4. Theoretical arrival times

Theoretical P and S wave arrival times are calculated using a simple 1D model:

$$t = t_0 + \frac{d}{v}$$

where $t_0$ is the event origin time (set to 0), $d$ is the epicentral distance, and $v$ is the wave velocity (P or S). These theoretical times provide initial estimates for the onset detection search windows.

In [5]:
logger.info("Calculating theoretical arrival times...")

# Add theoretical arrivals (t_p_theo, t_s_theo)
df_meta_stations = add_theoretical_arrivals(
    df_meta_stations,
    origin_time=0,
    distance_col='EPICENTRAL_DISTANCE_KM'
)

check('t_p_theo' in df_meta_stations.columns, "t_p_theo column added")
check('t_s_theo' in df_meta_stations.columns, "t_s_theo column added")

# Display results
print("\nTheoretical arrivals (first 5 stations):")
display(df_meta_stations[[
    'STATION_CODE',
    'EPICENTRAL_DISTANCE_KM',
    't_p_theo',
    't_s_theo'
]].head())

# Summary
print("\nTheoretical arrival time ranges:")
print(f"  P-wave: {df_meta_stations['t_p_theo'].min():.2f} - {df_meta_stations['t_p_theo'].max():.2f} s")
print(f"  S-wave: {df_meta_stations['t_s_theo'].min():.2f} - {df_meta_stations['t_s_theo'].max():.2f} s")

INFO | Calculating theoretical arrival times...
INFO | t_p_theo column added
INFO | t_s_theo column added


Added theoretical arrival times
t_P: 0.76 - 16.32 s
t_S: 1.31 - 28.89 s

Theoretical arrivals (first 5 stations):


,STATION_CODE,EPICENTRAL_DISTANCE_KM,t_p_theo,t_s_theo
0,EILF,109.5,16.271354,28.892919
3,ESCA,86.5,12.853627,22.824086
6,ISO,39.8,6.286352,10.862783
9,MFC,60.9,9.619533,16.622380
12,MON,98.2,14.592210,25.911275



Theoretical arrival time ranges:
  P-wave: 0.76 - 16.32 s
  S-wave: 1.31 - 28.89 s


## 5. Signal conversion

The acceleration DataFrame in long format (one row per sample) is converted to a nested dictionary structure for efficient access during onset detection:

```python
signals_dict = {
    'SURF': {
        'HNE': array([...]),  # East component
        'HNN': array([...]),  # North component
        'HNZ': array([...]),  # Vertical component
        'time': array([...])  # Time array (shared)
    },
    'BRZ': {...},
    ...
}
```

In [6]:
logger.info("Converting signals to nested dictionary...")

# Convert DataFrame to dict
signals_dict = convert_signals_to_dict(df_signals)

check(len(signals_dict) > 0, "Signals dictionary created")
logger.info(f"Dictionary contains {len(signals_dict)} stations")

# Validate structure
print("\nValidating signals dictionary...")
report = validate_signals_dict(signals_dict)

check(report['valid'], "All signals validated successfully")

INFO | Converting signals to nested dictionary...


Converting 66 files to nested dictionary...


KeyError: 'time'

In [7]:
print("Columns in df_signals:")
print(df_signals.columns.tolist())
print("\nFirst few rows:")
print(df_signals.head())

Columns in df_signals:
['file', 'sample', 'acceleration']

First few rows:
                                       file  sample  acceleration
0  FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC       0 -6.666667e-10
1  FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC       1 -6.666667e-10
2  FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC       2 -6.666667e-10
3  FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC       3 -6.666667e-10
4  FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC       4 -6.666667e-10
